In [ ]:

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
import csv, os, time, random, re

# ================= CONFIG =================
BASE_URL = "https://www.udemy.com/courses/search/?q=machine+learning&src=ukw&p="
OUT_CSV = "udemy_courses_machine_learning.csv"
HEADLESS = False
WAIT_TIMEOUT = 15
MAX_PAGES = 417  


# ================= HELPERS =================
def random_sleep(a=1.0, b=2.0):
    time.sleep(random.uniform(a, b))


def scroll_to_bottom(driver, pause=1.0):
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height


def get_text_by_selectors(driver, selectors):
    for sel in selectors:
        try:
            el = driver.find_element(By.CSS_SELECTOR, sel)
            if el.text.strip():
                return el.text.strip()
        except:
            continue
    return ""


def extract_rating_number(rating_text):
    if not rating_text:
        return ""
    match = re.search(r'(\d+\.\d+|\d+)', rating_text.replace(',', ''))
    return match.group(1) if match else ""


def extract_num_reviews(reviews_text):
    if not reviews_text:
        return ""
    match = re.search(r'\(([\d,]+)\)', reviews_text)
    return match.group(1) if match else ""


def extract_num_students(students_text):
    if not students_text:
        return ""
    match = re.search(r'([\d,]+)', students_text)
    return match.group(1) if match else ""


def extract_discount(discount_text):
    if not discount_text:
        return ""
    match = re.search(r'(\d+)%', discount_text)
    return f"{match.group(1)}%" if match else ""


def get_instructor(driver):
    try:
        els = driver.find_elements(By.CSS_SELECTOR, "span.instructor-links--names--fJWIai span.ud-btn-label")
        if els and els[0].text.strip():
            return els[0].text.strip().split()[0]
    except:
        pass
    try:
        el = driver.find_element(By.CSS_SELECTOR, "a[href^='#instructor-'] span.ud-btn-label")
        if el and el.text.strip():
            return el.text.strip().split()[0]
    except:
        pass
    return ""


def get_related_topics(driver):
    try:
        els = driver.find_elements(
            By.CSS_SELECTOR,
            "div.topic-navigation-module--topic-navigation--wCbdV ul li a"
        )
        topics = [e.text.strip() for e in els if e.text.strip()]
        return ", ".join(topics)
    except:
        return ""


def get_curriculum_info(driver):
    try:
        el = driver.find_element(By.CSS_SELECTOR, "div.ud-text-sm[data-purpose='curriculum-stats']")
        return el.text.strip()
    except:
        return ""


def get_price_info(driver):
    price_selectors = [
        "div[data-purpose='course-price-text']",
        "span.price-text--price-part--Tu6MH",
        "div.ud-heading-xl",
        "span.ud-sr-only"
    ]
    discount_selectors = [
        "div[data-purpose='discount-percentage']",
        "span.discount-percentage",
        "div.discount-badge"
    ]
    price, discount = "", ""
    for selector in price_selectors:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, selector)
            for e in els:
                text = e.text.strip()
                if text and any(ch.isdigit() for ch in text):
                    price = text
                    break
            if price:
                break
        except:
            continue
    for selector in discount_selectors:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, selector)
            for e in els:
                text = e.text.strip()
                if text and '%' in text:
                    discount = extract_discount(text)
                    break
            if discount:
                break
        except:
            continue
    return price, discount


def ensure_csv_header(path, fieldnames):
    if not os.path.exists(path):
        with open(path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()


def append_csv(path, row, fieldnames):
    with open(path, "a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)


def crawl_course(driver, url):
    driver.get(url)
    random_sleep(3, 5)
    scroll_to_bottom(driver, pause=1.2)
    random_sleep(2, 3)

    title = get_text_by_selectors(driver, ["h1[data-purpose='lead-title']", "h1.ud-heading-xxl"])
    headline = get_text_by_selectors(driver, ["div[data-purpose='lead-headline']", "div.ud-text-lg"])
    instructor = get_instructor(driver)
    rating = extract_rating_number(get_text_by_selectors(driver, ["span[data-purpose='rating-number']", "div.rating-number"]))
    num_reviews = extract_num_reviews(get_text_by_selectors(driver, ["button[data-purpose='rating']", "span.ud-text-xs"]))
    num_students = extract_num_students(get_text_by_selectors(driver, ["div[data-purpose='enrollment']", "span.students-count"]))
    language = get_text_by_selectors(driver, ["div[data-purpose='lead-course-locale']", "div.clp-lead__locale"])
    price, discount = get_price_info(driver)
    bestseller = get_text_by_selectors(driver, ["div[data-purpose='badge']", "div.ribbon-module--ribbon--", "div.bestseller-badge"])
    related_topics = get_related_topics(driver)
    curriculum_info = get_curriculum_info(driver)

    row = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "course_url": url,
        "title": title,
        "headline": headline,
        "is_bestseller": "Yes" if bestseller else "No",
        "rating": rating,
        "num_reviews": num_reviews,
        "num_students": num_students,
        "instructor": instructor,
        "language": language,
        "price": price,
        "discount": discount,
        "related_topics": related_topics,
        "curriculum_info": curriculum_info
    }
    return row


# ================= MAIN LOOP =================
opts = uc.ChromeOptions()
if HEADLESS:
    opts.add_argument("--headless=new")
opts.add_argument("--start-maximized")
opts.add_argument("--lang=en-US")

driver = uc.Chrome(options=opts)

print(f"🔍 Bắt đầu crawl Udemy theo trang (total = {MAX_PAGES})")

for page_num in range(130, MAX_PAGES + 1):
    page_url = BASE_URL + str(page_num)
    print(f"\n📄 Đang crawl trang {page_num} → {page_url}")
    driver.get(page_url)
    random_sleep(4, 6)
    scroll_to_bottom(driver, pause=1.5)
    random_sleep(2, 3)

    course_links = []
    els = driver.find_elements(By.CSS_SELECTOR, "a.ud-link-neutral.ud-custom-focus-visible")
    for e in els:
        try:
            href = e.get_attribute("href")
            if href and "/course/" in href and href not in course_links:
                course_links.append(href.split("?")[0])
        except:
            continue

    print(f"🔗 Tìm thấy {len(course_links)} khóa học trong trang {page_num}")

    for idx, link in enumerate(course_links, start=1):
        print(f"   → ({idx}/{len(course_links)}) {link}")
        try:
            row = crawl_course(driver, link)
            ensure_csv_header(OUT_CSV, row.keys())
            append_csv(OUT_CSV, row, row.keys())
        except Exception as e:
            print(f"⚠️  Lỗi khi crawl {link}: {e}")
        random_sleep(2, 4)
        driver.back()
        random_sleep(2, 3)

print("\n✅ Hoàn thành! Dữ liệu lưu tại:", OUT_CSV)
driver.quit()


🔍 Bắt đầu crawl Udemy theo trang (total = 417)

📄 Đang crawl trang 130 → https://www.udemy.com/courses/search/?q=machine+learning&src=ukw&p=130
🔗 Tìm thấy 26 khóa học trong trang 130
   → (1/26) https://www.udemy.com/course/data-analysis-and-data-science/
   → (2/26) https://www.udemy.com/course/python-data-visualization-with-matplotlib-2x/
   → (3/26) https://www.udemy.com/course/learning-path-python-effective-data-analysis-using-python/
   → (4/26) https://www.udemy.com/course/aws-machine-learning/
   → (5/26) https://www.udemy.com/course/aws-machine-learning-practice-exam/
   → (6/26) https://www.udemy.com/course/automlgoogle/
   → (7/26) https://www.udemy.com/course/30-days-of-python-code-numpy-challenge/
   → (8/26) https://www.udemy.com/course/complete-mathematics-for-data-science-business-analytics/
   → (9/26) https://www.udemy.com/course/enterprise-model-governance-risk-management/
   → (10/26) https://www.udemy.com/course/visualization-with-python-matplotlib-all-in-one/
   → 

KeyboardInterrupt: 